## US Cities Transportation and Walkability Analysis

    "This notebook analyzes data on transportation emissions and walkability in US cities. The goal is to explore relationships between vehicle usage, greenhouse gas emissions, and walkability indices.\n",

## Dataset Overview
- **full_city_data_DATA603.csv**: Aggregated city-level data including population, housing, miles driven, fuel consumption, emissions, and walkability metrics.
- **city_blockgroup_pairs_DATA603.csv**: Mapping between cities and census block groups.
- **Cleaned versions**: Processed data with invalid values handled.

## Objectives
1. Load and clean the data
2. Perform exploratory data analysis
3. Visualize key metrics
4. Analyze correlations between emissions and walkability
5. Identify cities with high emissions and low walkability

## 1. Install and Import Dependencies

If running on Google Colab, uncomment the installation lines if needed.

In [1]:
# Install dependencies (uncomment if needed)
# !pip install pyspark pandas numpy matplotlib seaborn plotly

# Import libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Start Spark and Load Data

Use PySpark to load the full dataset and keep the project aligned with big data processing.

In [ ]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName('MontanaWalkabilityAnalysis') \
    .config('spark.driver.memory', '2g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .getOrCreate()

# Load the cleaned full city data using Spark
spark_df = spark.read.csv('cleaned_full_city_data.csv', header=True, inferSchema=True)

# Filter to US cities dataset
df = spark_df.filter(col('state_abbr') == 'MT')

print('Spark DataFrame schema:')
df.printSchema()
print('Record count:', df.count())
print('Sample rows:')
df.show(5, truncate=False)

## 3. Data Cleaning with Spark

Use Spark operations to clean invalid values and prepare features for modeling.

In [ ]:
# Replace invalid sentinel values with nulls
clean_df = df.replace([-99999, -99998], [None, None])

# Define numeric columns and cast them explicitly
numeric_cols = ['population', 'house_units', 'miles_driven_pC', 'agg_vehi_consum_pC',
                'gasoline_consume_pC', 'diesel_consume_pC', 'agg_vehi_emitGHG_pC',
                'agg_nonvehi_emitGHG_pC', 'agg_combined_emitGHG_pC', 'driveGasoline_emitGHG_pC',
                'driveDiesel_emitGHG_pC', 'wlk_NatWalkInd_avg', 'wlk_PeopleCounted',
                'wlk_Workers', 'wlk_CountHU', 'wlk_HH', 'wlk_D2A_EPHHM_avg',
                'wlk_D2B_E8MIXA_avg', 'wlk_D3B_avg', 'wlk_D4A_avg', 'wlk_D2A_Ranked_avg',
                'wlk_D2B_Ranked_avg', 'wlk_D3B_Ranked_avg', 'wlk_D4A_Ranked_avg']

for col_name in numeric_cols:
    clean_df = clean_df.withColumn(col_name, col(col_name).cast('double'))

# Drop rows missing the target variable or key features for modeling
clean_df = clean_df.na.drop(subset=['agg_combined_emitGHG_pC', 'miles_driven_pC', 'wlk_NatWalkInd_avg'])

print('Cleaned Spark DataFrame schema:')
clean_df.printSchema()
print('Missing values count by column:')
for col_name in numeric_cols:
    null_count = clean_df.filter(col(col_name).isNull()).count()
    if null_count > 0:
        print(f'  {col_name}: {null_count}')

print('Remaining record count:', clean_df.count())

## 4. Exploratory Data Analysis

Convert the cleaned Spark DataFrame to pandas for summary statistics and plots.

In [ ]:
# Collect the cleaned data to pandas for plotting
pandas_df = clean_df.toPandas()

print('Basic statistics for key variables:')
key_vars = ['population', 'miles_driven_pC', 'agg_combined_emitGHG_pC', 'wlk_NatWalkInd_avg']
print(pandas_df[key_vars].describe())

# Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Distributions of Key Variables')

sns.histplot(pandas_df['population'], ax=axes[0,0], kde=True)
axes[0,0].set_title('Population Distribution')

sns.histplot(pandas_df['miles_driven_pC'], ax=axes[0,1], kde=True)
axes[0,1].set_title('Miles Driven per Capita')

sns.histplot(pandas_df['agg_combined_emitGHG_pC'], ax=axes[1,0], kde=True)
axes[1,0].set_title('Combined GHG Emissions per Capita')

sns.histplot(pandas_df['wlk_NatWalkInd_avg'], ax=axes[1,1], kde=True)
axes[1,1].set_title('National Walkability Index')

plt.tight_layout()
plt.savefig('figures/distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

Use Spark for data processing and pandas for correlation visualization.

In [ ]:
# Create a small correlation data subset
corr_cols = ['miles_driven_pC', 'agg_vehi_emitGHG_pC', 'agg_nonvehi_emitGHG_pC',
             'agg_combined_emitGHG_pC', 'wlk_NatWalkInd_avg', 'wlk_D2A_EPHHM_avg',
             'wlk_D2B_E8MIXA_avg', 'population']

corr_df = pandas_df[corr_cols].dropna()

corr_matrix = corr_df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix: Emissions and Walkability')
plt.savefig('figures/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter plots for key relationships
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.scatterplot(data=pandas_df, x='wlk_NatWalkInd_avg', y='miles_driven_pC', ax=axes[0])
axes[0].set_title('Miles Driven vs National Walkability Index')
axes[0].set_xlabel('National Walkability Index')
axes[0].set_ylabel('Miles Driven per Capita')

sns.scatterplot(data=pandas_df, x='wlk_NatWalkInd_avg', y='agg_combined_emitGHG_pC', ax=axes[1])
axes[1].set_title('GHG Emissions vs National Walkability Index')
axes[1].set_xlabel('National Walkability Index')
axes[1].set_ylabel('Combined GHG Emissions per Capita')

sns.scatterplot(data=pandas_df, x='population', y='agg_combined_emitGHG_pC', ax=axes[2])
axes[2].set_title('Population vs GHG Emissions')
axes[2].set_xlabel('Population')
axes[2].set_ylabel('Combined GHG Emissions per Capita')

plt.tight_layout()
plt.savefig('figures/scatter_plots.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Spark Machine Learning: Linear Regression

Build and evaluate a Spark ML linear regression model to predict combined emissions per capita.

In [ ]:
# Prepare features for Spark ML
feature_cols = ['miles_driven_pC', 'wlk_NatWalkInd_avg', 'population']
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
model_df = assembler.transform(clean_df).select('features', 'agg_combined_emitGHG_pC')

# Split the data into train and test sets
train_df, test_df = model_df.randomSplit([0.75, 0.25], seed=42)

lr = LinearRegression(labelCol='agg_combined_emitGHG_pC', featuresCol='features')
lr_model = lr.fit(train_df)

print('Linear Regression Model Coefficients:', lr_model.coefficients)
print('Linear Regression Model Intercept:', lr_model.intercept)

# Evaluate model performance
predictions = lr_model.transform(test_df)
evaluator_rmse = RegressionEvaluator(labelCol='agg_combined_emitGHG_pC', predictionCol='prediction', metricName='rmse')
evaluator_r2 = RegressionEvaluator(labelCol='agg_combined_emitGHG_pC', predictionCol='prediction', metricName='r2')

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)
print(f'RMSE: {rmse:.4f}')
print(f'R2: {r2:.4f}')

# Show prediction sample
predictions.select('prediction', 'agg_combined_emitGHG_pC', 'features').show(10, truncate=False)

## 7. Model Interpretation and Big Data Conclusion

Summarize the machine learning results and how Spark supports big data workflows.

In [ ]:
# Save processed data and model outputs
clean_df.write.mode('overwrite').parquet('processed_city_data.parquet')
print('Saved cleaned data in Parquet format for big data reuse.')

results_pdf = predictions.select('prediction', 'agg_combined_emitGHG_pC', 'features').toPandas()
results_pdf.head()

## 8. Summary and Conclusions

This notebook now uses a big data processing pipeline with Spark for data loading, cleaning, transformation, and machine learning.

- PySpark is used to load and clean the dataset at scale.
- Spark ML linear regression predicts combined greenhouse gas emissions per capita.
- The project supports reproducible big-data analytics and can scale beyond state-level US cities data.

In [ ]:
# Stop Spark session
spark.stop()
print('Spark session stopped. Analysis complete.')